# External comparison — BhashaBench-Legal

Every number in this project so far comes from a benchmark we built ourselves.
That is a real weakness: our own scorer had four bugs, and a metric you wrote
cannot tell you whether you are competitive with anyone else.

BhashaBench-Legal is multiple-choice, so scoring is exact — no phrase matching,
no partial credit, nothing for us to get wrong. It is the one number here that
is directly comparable against any other model.

**Settings:** GPU **T4 x2**, Internet **On**, `HF_TOKEN` secret set (the dataset
is gated). ~40 min.

In [ ]:
!pip -q install -U transformers accelerate peft datasets

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"     # T4 x2 -> DataParallel breaks placement
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import json, re, time
import torch

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as exc:
    raise RuntimeError(f"HF_TOKEN secret required — the dataset is gated: {exc}")

major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
print(f"GPU: {torch.cuda.get_device_name(0)} ({arch})")
if arch not in torch.cuda.get_arch_list():
    raise RuntimeError(f"{arch} unsupported by this torch build")

# Native bf16 only on Ampere+. torch.cuda.is_bf16_supported() defaults to
# including_emulation=True and says True on a T4, where bf16 is software-
# emulated and several times slower.
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("dtype:", DTYPE)

In [ ]:
from datasets import concatenate_datasets, load_dataset

# Verified schema: option_a..option_d / correct_answer, and separate English
# and Hindi configs — there is no single "test" split.
SUBSET = 750          # per language; keeps the run inside ~40 min
parts = []
for config in ("English", "Hindi"):
    ds = load_dataset("bharatgenai/BhashaBench-Legal", config, split="test")
    if SUBSET and len(ds) > SUBSET:
        ds = ds.shuffle(seed=0).select(range(SUBSET))
    parts.append(ds)
    print(f"{config}: {len(ds)}")
bench = concatenate_datasets(parts)
print(len(bench), "questions |", bench.column_names)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LETTERS = ("A", "B", "C", "D")

def mcq_prompt(row):
    opts = "\n".join(f"{L}. {row['option_' + L.lower()]}" for L in LETTERS
                     if row.get("option_" + L.lower()) not in (None, ""))
    return (f"{row['question']}\n{opts}\n\n"
            "Answer with the single letter of the correct option.")

def run_mcq(model_id, label, batch_size=16):
    tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=DTYPE, device_map={"": 0}).eval()

    correct = total = unparsed = 0
    rows, t0 = [], time.time()
    for start in range(0, len(bench), batch_size):
        batch = bench.select(range(start, min(start + batch_size, len(bench))))
        texts = [tok.apply_chat_template([{"role": "user", "content": mcq_prompt(r)}],
                                         tokenize=False, add_generation_prompt=True)
                 for r in batch]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=8, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        for row, text in zip(batch, tok.batch_decode(
                out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            m = re.search(r"\b([ABCD])\b", text.upper())
            pred = m.group(1) if m else None
            unparsed += pred is None
            gold = str(row["correct_answer"]).strip().upper()[:1]
            correct += pred == gold
            total += 1
            rows.append({"pred": pred, "gold": gold,
                         "language": row.get("language"),
                         "domain": row.get("subject_domain")})
        if total % 160 == 0:
            print(f"  {label}: {total}/{len(bench)} acc={correct/total:.1%} "
                  f"({time.time()-t0:.0f}s)", flush=True)

    del model
    torch.cuda.empty_cache()
    with open(f"/kaggle/working/bhashabench_{label}.jsonl", "w") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")

    by_lang = {}
    for lang in {r["language"] for r in rows}:
        sub = [r for r in rows if r["language"] == lang]
        by_lang[lang] = sum(r["pred"] == r["gold"] for r in sub) / len(sub)
    # Unparsed replies are reported separately: a model ignoring the answer
    # format is a different failure from one that does not know the law, and
    # folding them together would understate it.
    print(f"{label}: {correct}/{total} = {correct/total:.1%} "
          f"(unparseable: {unparsed}) by-language "
          f"{ {k: f'{v:.1%}' for k, v in by_lang.items()} }")
    return {"accuracy": correct / total, "n": total,
            "unparsed": unparsed, "by_language": by_lang}

scores = {}
for model_id, label in [("Qwen/Qwen2.5-3B-Instruct", "base"),
                        ("NyayaLabs98/nyaya-3b-v3", "nyaya-3b-v3")]:
    scores[label] = run_mcq(model_id, label)

json.dump(scores, open("/kaggle/working/bhashabench_scores.json", "w"), indent=2)
print(json.dumps(scores, indent=2))

In [ ]:
# Is the difference real? MCQ accuracy on n questions has a standard error of
# sqrt(p(1-p)/n); a gap smaller than ~2 SE is noise. Same discipline as the
# paired bootstrap used on our own benchmark.
import math

b, v = scores["base"], scores["nyaya-3b-v3"]
se = math.sqrt(b["accuracy"] * (1 - b["accuracy"]) / b["n"]
               + v["accuracy"] * (1 - v["accuracy"]) / v["n"])
delta = v["accuracy"] - b["accuracy"]
print(f"base {b['accuracy']:.1%}  nyaya-3b-v3 {v['accuracy']:.1%}")
print(f"delta {delta:+.2%}  95% CI [{delta-1.96*se:+.2%}, {delta+1.96*se:+.2%}]")
print("TIED — CI spans zero" if abs(delta) < 1.96 * se else "REAL DIFFERENCE")
print("\nRandom guessing on 4 options is 25%. Anything near that means the")
print("model is not answering the format, not that it knows no law.")